# Desserializando por nome do campo ou alias

Vimos que, ao desserializar, campos com alias definido precisam usar o alias nos dados.

Podemos mudar isso para aceitar tanto o alias quanto o nome do campo.

Fazemos isso configurando o modelo com `model_config`.

In [1]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError

Primeiro, confirmando que desserializar pelo nome do campo, por padrão, não funciona:

In [2]:
class Model(BaseModel):
    id_: int = Field(alias="id")
    first_name: str = Field(alias="firstName")

In [3]:
try:
    Model(id_=10, first_name="Newton")
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 2 validation errors for Model
id
  Field required [type=missing, input_value={'id_': 10, 'first_name': 'Newton'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
firstName
  Field required [type=missing, input_value={'id_': 10, 'first_name': 'Newton'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


O mesmo ao desserializar um dict ou JSON:

In [4]:
data = {
    "id_": 10,
    "first_name": "Newton"
}

try:
    Model.model_validate(data)
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 2 validation errors for Model
id
  Field required [type=missing, input_value={'id_': 10, 'first_name': 'Newton'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
firstName
  Field required [type=missing, input_value={'id_': 10, 'first_name': 'Newton'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


Configurando o modelo para permitir isso:

In [5]:
class Model(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    id_: int = Field(alias="id")
    first_name: str = Field(alias="firstName")

Agora podemos usar **tanto** o nome do campo quanto o alias:

In [6]:
print(f'Model(id_=10, first_name="Newton"): {Model(id_=10, first_name="Newton")!r}')

Model(id_=10, first_name="Newton"): Model(id_=10, first_name='Newton')


Dá até para misturar:

In [7]:
print(f'Model(id_=10, firstName="Newton"): {Model(id_=10, firstName="Newton")!r}')

Model(id_=10, firstName="Newton"): Model(id_=10, first_name='Newton')


E funciona também na desserialização de dict ou JSON:

In [8]:
data = {
    "id_": 10,
    "first_name": "Newton"
}

print(f"Model.model_validate(data): {Model.model_validate(data)!r}")

Model.model_validate(data): Model(id_=10, first_name='Newton')


Recapitulando o que temos para configurar aliases e comportamentos associados:

In [9]:
from pydantic.alias_generators import to_camel

class Person(BaseModel):
    model_config = ConfigDict(
        alias_generator=to_camel,
        populate_by_name=True,
        extra="forbid"
    )

    id_: int = Field(alias="id", default=1)
    first_name: str | None = None
    last_name: str
    age: int | None = None

In [10]:
p = Person(id=10, first_name='Isaac', lastName='Newton', age=84)
print(f"p: {p!r}")

p: Person(id_=10, first_name='Isaac', last_name='Newton', age=84)


In [11]:
data_json = """
{
    "id": 10,
    "firstName": "Isaac",
    "last_name": "Newton",
    "age": 84
}
"""

p = Person.model_validate_json(data_json)
print(f"p: {p!r}")

p: Person(id_=10, first_name='Isaac', last_name='Newton', age=84)


In [12]:
print(f"p.model_dump(): {p.model_dump()!r}")

p.model_dump(): {'id_': 10, 'first_name': 'Isaac', 'last_name': 'Newton', 'age': 84}


Repare que `populate_by_name` não afeta os nomes usados na **serialização** — o padrão continua sendo o nome do campo, não o alias.

É preciso pedir explicitamente:

In [13]:
print(f"p.model_dump(by_alias=True): {p.model_dump(by_alias=True)!r}")

p.model_dump(by_alias=True): {'id': 10, 'firstName': 'Isaac', 'lastName': 'Newton', 'age': 84}


In [14]:
print(f"p.model_dump_json(by_alias=True): {p.model_dump_json(by_alias=True)!r}")

p.model_dump_json(by_alias=True): '{"id":10,"firstName":"Isaac","lastName":"Newton","age":84}'


> 📝 **Nota de revisão (Pydantic 2.11+):** a documentação atual recomenda trocar `populate_by_name=True` pelo par
> `validate_by_alias=True` + `validate_by_name=True` (mesmo efeito, nome mais claro; `populate_by_name` está marcado para depreciação na v3).
> E existe `serialize_by_alias=True`, que faz o `model_dump()` usar aliases sem precisar de `by_alias=True` a cada chamada:

In [15]:
class PersonModern(BaseModel):
    model_config = ConfigDict(
        alias_generator=to_camel,
        validate_by_alias=True,
        validate_by_name=True,
        serialize_by_alias=True,
        extra="forbid",
    )

    id_: int = Field(alias="id", default=1)
    first_name: str | None = None
    last_name: str
    age: int | None = None

In [16]:
p = PersonModern.model_validate_json(data_json)
print(f"p.model_dump(): {p.model_dump()!r}")

p.model_dump(): {'id': 10, 'firstName': 'Isaac', 'lastName': 'Newton', 'age': 84}
